# What does `jax.jacfwd` do? A small demo

1. A toy function: what forward-mode autodiff computes and how.
2. The same thing on the N-body package: derivatives of TRAPPIST-1 transit times with respect to the
   planet masses, compared with the package's own derivatives (`dtdq0`) and with finite differences.

In [1]:
import sys
sys.path.insert(0, "..")          # the package (src/) lives one folder up

import src                        # import the package before running anything in jax (CPU runtime setup)
from src import Integrator, Integrator_no_grad, State, get_default_ICs
import jax
import jax.numpy as jnp
import numpy as np
from dataclasses import replace
import time

## 1. Toy example

$f(x, y) = \big(x^2 y,\; \sin x + y\big)$. Its Jacobian (rows = outputs, columns = inputs) is

$$J = \begin{pmatrix} \partial f_1/\partial x & \partial f_1/\partial y \\ \partial f_2/\partial x & \partial f_2/\partial y \end{pmatrix}
     = \begin{pmatrix} 2xy & x^2 \\ \cos x & 1 \end{pmatrix}$$

In [2]:
def f(p):
    x, y = p
    return jnp.array([x**2 * y, jnp.sin(x) + y])

p = jnp.array([1.5, 2.0])
x, y = p

J_auto = jax.jacfwd(f)(p)
J_hand = jnp.array([[2 * x * y, x**2],
                    [jnp.cos(x), 1.0]])
print("jax.jacfwd:\n", J_auto)
print("by hand:\n", J_hand)
print("max difference:", float(jnp.max(jnp.abs(J_auto - J_hand))))

jax.jacfwd:
 [[6.        2.25     ]
 [0.0707372 1.       ]]
by hand:
 [[6.        2.25     ]
 [0.0707372 1.       ]]
max difference: 0.0


### How forward mode gets there: `jax.jvp`

Forward mode never builds formulas. It runs `f` once while carrying, next to every number, a **tangent**:
how that number changes when the input moves in a chosen direction `v`. Every operation updates both
(e.g. for `a * b`: value `a*b`, tangent `ȧ*b + a*ḃ` — the chain rule, applied number by number).

`jax.jvp(f, (p,), (v,))` returns `f(p)` and `J @ v` — one column of the Jacobian per direction —
from a single run of `f`:

In [3]:
value, dfdx = jax.jvp(f, (p,), (jnp.array([1.0, 0.0]),))   # direction: move x only
_,     dfdy = jax.jvp(f, (p,), (jnp.array([0.0, 1.0]),))   # direction: move y only
print("f(p)            =", value)
print("J @ [1, 0] (d/dx) =", dfdx, "  = column 0 of J")
print("J @ [0, 1] (d/dy) =", dfdy, "  = column 1 of J")

f(p)            = [4.5        2.99749499]
J @ [1, 0] (d/dx) = [6.        0.0707372]   = column 0 of J
J @ [0, 1] (d/dy) = [2.25 1.  ]   = column 1 of J


So **`jax.jacfwd` = one `jvp` per input direction**, batched together. Its cost grows with the
number of *inputs* (here 2). Reverse mode (`jax.grad`, `jax.jacrev`) instead costs per *output*.

In [4]:
# Compare with finite differences: approximate, and the error depends on the step eps
for eps in [1e-2, 1e-5, 1e-8, 1e-11]:
    J_fd = jnp.stack([(f(p + eps * e) - f(p - eps * e)) / (2 * eps) for e in jnp.eye(2)], axis=1)
    print(f"finite difference, eps = {eps:.0e}: max error {float(jnp.max(jnp.abs(J_fd - J_hand))):.1e}")

finite difference, eps = 1e-02: max error 1.2e-06
finite difference, eps = 1e-05: max error 3.9e-11
finite difference, eps = 1e-08: max error 3.6e-08
finite difference, eps = 1e-11: max error 2.2e-05


Finite differences are limited by truncation error (eps too large) and round-off (eps too small).
Autodiff is exact to round-off, with no step to tune.

## 2. The same on the N-body problem

Wrap "planet masses → transit times" as an ordinary function, using the derivative-free
`Integrator_no_grad`, and let `jax.jacfwd` differentiate *through the whole integration*:
every AHL21 step, Kepler solve and transit search. 8 inputs (masses) → 8 tangents.

A short run (`tmax = 30` days) keeps compilation quick.

In [5]:
tmax = 30.0
h = 0.1
ic = get_default_ICs("trappist-1")
s0 = State.from_ic(ic)
tt0 = src.create_transit_timing(tmax=tmax, ic=ic)
intr = Integrator_no_grad.default(h=h, t0=0.0, tmax=tmax)

def transit_times(m):
    """Masses (8,) -> transit times (8 bodies, ntt). Positions and velocities are kept fixed."""
    s = replace(s0, m=m)
    _, tt = intr(s, tt0)
    return tt.tt

t = time.perf_counter()
dt_dm = jax.jit(jax.jacfwd(transit_times))(s0.m)      # shape (8, ntt, 8): [planet, transit, mass of body]
jax.block_until_ready(dt_dm)
print(f"jacfwd (incl. compilation): {time.perf_counter() - t:.1f} s,  shape {dt_dm.shape}")

count = np.asarray(intr(s0, tt0)[1].count)
print("transits per planet:", count[1:])

jacfwd (incl. compilation): 22.6 s,  shape (8, 23, 8)


transits per planet: [20 13  7  4  3  2  2]


### Check 1: against the package's own derivatives

`Integrator` (the gradient integrator) propagates the Jacobian with respect to all initial
coordinates and masses. `dtdq0[planet, transit, k, body]` with `k = 0..5` for (x, y, z, vx, vy, vz)
and **`k = 6` for the mass**.

In [6]:
intr_grad = Integrator.default(h=h, t0=0.0, tmax=tmax)
_, tt_grad = intr_grad(State.from_ic(ic), tt0)
dt_dm_pkg = np.asarray(tt_grad.dtdq0)[:, :, 6, :]        # (8, ntt, 8)

worst = 0.0
for p in range(1, 8):
    a = np.asarray(dt_dm)[p, :count[p]]
    b = dt_dm_pkg[p, :count[p]]
    worst = max(worst, np.max(np.abs(a - b)) / np.max(np.abs(b)))
print(f"max relative difference, jacfwd vs package: {worst:.1e}")

max relative difference, jacfwd vs package: 5.7e-13


### Check 2: against finite differences, for one mass

Nudge planet c's mass up and down and look at how planet b's transit times move.

In [7]:
j, planet = 2, 1                       # mass of c, transit times of b
eps = 1e-6 * float(s0.m[j])
tp = np.asarray(transit_times(s0.m.at[j].add(eps)))
tm = np.asarray(transit_times(s0.m.at[j].add(-eps)))
fd = (tp - tm) / (2 * eps)

n = count[planet]
print("transit   jacfwd            package           finite difference   [days per unit mass, star = 1]")
for i in [0, n // 2, n - 1]:
    print(f"{i:5d}   {float(dt_dm[planet, i, j]): .8e}   {dt_dm_pkg[planet, i, j]: .8e}   {fd[planet, i]: .8e}")

transit   jacfwd            package           finite difference   [days per unit mass, star = 1]
    0   -8.27305585e-01   -8.27305585e-01   -8.27301569e-01
   10    3.84609275e+01    3.84609275e+01    3.84589337e+01
   19    5.86683370e+01    5.86683370e+01    5.86619071e+01


### What to take away

* `jax.jacfwd(fn)` gives the exact derivative of the *program* `fn`, here the whole N-body
  integration, by carrying one tangent per input through every operation.
* All three agree: jacfwd vs the package to ~1e-12, finite differences to their own (lower) accuracy.
* Why the package still has its own derivative code: it propagates the Jacobian with problem-specific
  structure (each Kepler pair touches only 2 bodies, the Kepler and transit solvers are differentiated
  at their converged answer), which is ~3–4× faster than generic `jacfwd` and compiles faster.
* `jax.grad` (reverse mode) does **not** work through this integrator: the Kepler solve and transit
  search are `lax.while_loop`s with data-dependent length, which JAX cannot run backwards.

In [8]:
try:
    jax.grad(lambda m: jnp.sum(transit_times(m)))(s0.m)
except ValueError as e:
    print("jax.grad failed:", str(e).splitlines()[0])

jax.grad failed: Reverse-mode differentiation does not work for lax.while_loop or lax.fori_loop with dynamic start/stop values. Try using lax.scan, or using fori_loop with static start/stop.
